In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN

In [34]:
data = pd.read_csv('/Users/valter.rebelo/MissionControl/data/micro/candleData/ripple_candles.csv')
data.dropna(inplace=True)
data = data[data['date'] > '2023-01-01']
volume = pd.read_csv('/Users/valter.rebelo/MissionControl/data/micro/assetData/ripple.csv')


In [52]:
data

# Calculate the current drawdown support (lowest close during current drawdown)
# First identify peaks (local maxima)
data['peak'] = data['close'].rolling(window=10, center=True).apply(lambda x: x[1] == max(x), raw=True)
data['peak'] = data['peak'].fillna(False)

# For each point, find the most recent peak
data['last_peak_idx'] = np.nan
last_peak_idx = -1
for i in range(len(data)):
    if data['peak'].iloc[i]:
        last_peak_idx = i
    data.loc[data.index[i], 'last_peak_idx'] = last_peak_idx

# Calculate the lowest close since the last peak (current drawdown support)
data['drawdown_support'] = np.nan
for i in range(len(data)):
    peak_idx = int(data['last_peak_idx'].iloc[i])
    if peak_idx >= 0:
        data.loc[data.index[i], 'drawdown_support'] = data['close'].iloc[peak_idx:i+1].min()

# Identify when a new drawdown begins (when we hit a new peak)
data['new_drawdown'] = False
for i in range(1, len(data)):
    if data['last_peak_idx'].iloc[i] != data['last_peak_idx'].iloc[i-1]:
        data.loc[data.index[i], 'new_drawdown'] = True

# Find previous drawdown supports
previous_supports = []
current_support = None
current_peak_idx = None

for i in range(len(data)):
    if data['new_drawdown'].iloc[i] and current_support is not None:
        # When a new drawdown begins, store the previous support
        previous_supports.append((current_peak_idx, current_support))
        current_support = None
    
    if data['peak'].iloc[i]:
        current_peak_idx = i
    
    if current_peak_idx is not None and i > current_peak_idx:
        # Update the current support level
        current_support = data['drawdown_support'].iloc[i]

# Plot the data using plotly
import plotly.graph_objects as go

fig = go.Figure()

# Add close price with alpha 0.4
fig.add_trace(go.Scatter(
    x=data.index,
    y=data['close'],
    mode='lines',
    name='Close Price',
    line=dict(color='blue'),
    opacity=0.4
))

# Add previous drawdown supports as horizontal lines
for i, (peak_idx, support_price) in enumerate(previous_supports):
    # Find the next peak index to determine where this support line should end
    next_peak_idx = None
    for j, (idx, _) in enumerate(previous_supports):
        if j > i and idx > peak_idx:
            next_peak_idx = idx
            break
    
    # If no next peak found, use the end of the data
    end_idx = next_peak_idx if next_peak_idx is not None else len(data) - 1
    
    # Create a horizontal line for previous support
    fig.add_trace(go.Scatter(
        x=[data.index[peak_idx], data.index[end_idx]],
        y=[support_price, support_price],
        mode='lines',
        name='Previous Support',
        line=dict(color='rgba(255, 165, 0, 1)', dash='dash', width=1),
        showlegend=i==0  # Only show legend once
    ))

# Add current drawdown support as a horizontal line
if current_support is not None:
    fig.add_trace(go.Scatter(
        x=[data.index[current_peak_idx], data.index[-1]],
        y=[current_support, current_support],
        mode='lines',
        name='Current Support',
        line=dict(color='rgba(255, 0, 0, 1)', dash='dash', width=2)
    ))

fig.update_layout(
    title='Bitcoin Price with Historical Support Levels',
    xaxis_title='Date',
    yaxis_title='Price',
    template='plotly_white',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()

In [100]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN

# Load your BTC candle data
candle_data = pd.read_csv('/Users/valter.rebelo/MissionControl/data/micro/candleData/bitcoin_candles.csv')
volume_data = pd.read_csv('/Users/valter.rebelo/MissionControl/data/micro/assetData/bitcoin.csv')

# Merge dataframes on 'date'
merged_data = pd.merge(candle_data, volume_data[['date', 'total_volume']], on='date', how='left')
merged_data.rename(columns={'total_volume': 'volume'}, inplace=True)
merged_data.dropna(inplace=True)
merged_data['date'] = pd.to_datetime(merged_data['date'])
merged_data.set_index('date', inplace=True)


merged_data['low_7'] = merged_data['low'].rolling(window=7).min()
merged_data['high_7'] = merged_data['high'].rolling(window=7).max()

merged_data['low_14'] = merged_data['low'].rolling(window=14).min()
merged_data['high_14'] = merged_data['high'].rolling(window=14).max()

merged_data['low_21'] = merged_data['low'].rolling(window=21).min()
merged_data['high_21'] = merged_data['high'].rolling(window=21).max()

merged_data['low_28'] = merged_data['low'].rolling(window=28).min()
merged_data['high_28'] = merged_data['high'].rolling(window=28).max()

merged_data['low_35'] = merged_data['low'].rolling(window=35).min()
merged_data['high_35'] = merged_data['high'].rolling(window=35).max()

merged_data['low_42'] = merged_data['low'].rolling(window=42).min()
merged_data['high_42'] = merged_data['high'].rolling(window=42).max()

merged_data['support_cluster'] = (merged_data['low_7']+merged_data['low_14']+merged_data['low_21']+merged_data['low_28']+merged_data['low_35']+merged_data['low_42'])/6
merged_data['resistance_cluster'] = (merged_data['high_7']+merged_data['high_14']+merged_data['high_21']+merged_data['high_28']+merged_data['high_35']+merged_data['high_42'])/6

# Calculate Bollinger Bands
# First, calculate the 20-day moving average
merged_data['ma30'] = merged_data['close'].rolling(window=30).mean()

# Calculate the standard deviation of the closing price over the same period
merged_data['std30'] = merged_data['close'].rolling(window=30).std()

# Calculate the upper and lower Bollinger Bands
# Upper band = 20-day MA + (20-day std * 2)
# Lower band = 20-day MA - (20-day std * 2)
merged_data['upper_band'] = merged_data['ma30'] + (merged_data['std30'] * 2)
merged_data['lower_band'] = merged_data['ma30'] - (merged_data['std30'] * 2)

# Calculate distance from bands
merged_data['distance_from_upper_band'] = merged_data['upper_band']/merged_data['close'] - 1
merged_data['distance_from_lower_band'] = merged_data['close']/merged_data['lower_band'] - 1

# Calculate rolling z-score normalization for distances (using 20-day window)
merged_data['distance_upper_band_zscore'] = (
    merged_data['distance_from_upper_band'] - 
    merged_data['distance_from_upper_band'].rolling(window=20).mean()
) / merged_data['distance_from_upper_band'].rolling(window=20).std()

merged_data['distance_lower_band_zscore'] = (
    merged_data['distance_from_lower_band'] - 
    merged_data['distance_from_lower_band'].rolling(window=20).mean()
) / merged_data['distance_from_lower_band'].rolling(window=20).std()

# Calculate log returns
merged_data['log_return'] = np.log(merged_data['close'] / merged_data['close'].shift(1))

# Calculate correlation between log returns and distances from bands
corr_upper_band = merged_data['log_return'].corr(merged_data['distance_upper_band_zscore'])
corr_lower_band = merged_data['log_return'].corr(merged_data['distance_lower_band_zscore'])

print(f"Correlation between log return and distance from upper band: {corr_upper_band:.4f}")
print(f"Correlation between log return and distance from lower band: {corr_lower_band:.4f}")



# Drop NaN values that result from the rolling calculations
merged_data.dropna(inplace=True)


Correlation between log return and distance from upper band: -0.4065
Correlation between log return and distance from lower band: 0.3549


In [101]:
merged_data['distance_from_lower_band']

date
2013-06-17    0.020703
2013-06-18    0.044135
2013-06-19    0.076581
2013-06-20    0.134238
2013-06-21    0.156667
                ...   
2025-03-09    0.039548
2025-03-10   -0.009437
2025-03-11   -0.015039
2025-03-12    0.044868
2025-03-13    0.065364
Name: distance_from_lower_band, Length: 4287, dtype: float64

In [113]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create a figure with subplots
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, 
                   vertical_spacing=0.1, 
                   subplot_titles=('Support and Resistance Clusters', 'Bollinger Bands'))

# Add close price to first subplot
fig.add_trace(
    go.Scatter(
        x=merged_data.index,
        y=merged_data['close'],
        mode='lines',
        name='Close Price',
        line=dict(color='#1f77b4', width=1.5)
    ),
    row=1, col=1
)

# Add support cluster to first subplot
fig.add_trace(
    go.Scatter(
        x=merged_data.index,
        y=merged_data['support_cluster'],
        mode='lines',
        name='Support Cluster',
        line=dict(color='#2ca02c', width=1.5)
    ),
    row=1, col=1
)

# Add resistance cluster to first subplot
fig.add_trace(
    go.Scatter(
        x=merged_data.index,
        y=merged_data['resistance_cluster'],
        mode='lines',
        name='Resistance Cluster',
        line=dict(color='#d62728', width=1.5)
    ),
    row=1, col=1
)



# Add upper bollinger band to second subplot
fig.add_trace(
    go.Scatter(
        x=merged_data.index,
        y=merged_data['distance_upper_band_zscore'],
        mode='lines',
        name='Upper Band',
        line=dict(color='#ff7f0e', width=1.5)
    ),
    row=2, col=1
)


# Add lower bollinger band to second subplot
fig.add_trace(
    go.Scatter(
        x=merged_data.index,
        y=merged_data['distance_lower_band_zscore'],
        mode='lines',
        name='Lower Band',
        line=dict(color='#8c564b', width=1.5)
    ),
    row=2, col=1
)

# Add lower bollinger band to second subplot
fig.add_trace(
    go.Scatter(
        x=merged_data.index,
        y=merged_data['bzs'],
        mode='lines',
        name='Lower Band',
        line=dict(color='#8c564b', width=1.5)
    ),
    row=3, col=1
)

# Update layout for a clean, elegant look
fig.update_layout(
    title='Bitcoin Price Analysis',
    template='plotly_white',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    ),
    margin=dict(l=40, r=40, t=80, b=40),
    hovermode='x unified',
    height=800
)

# Update y-axis labels
fig.update_yaxes(title_text="Price (USD)", row=1, col=1)
fig.update_yaxes(title_text="Price (USD)", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)

# Show the figure
fig.show()


In [111]:

merged_data['bzs'] = merged_data['distance_upper_band_zscore'] - merged_data['distance_lower_band_zscore']
merged_data['bzsd'] = merged_data['bzs'].pct_change()

In [109]:
merged_data['bzs'].std()

2.276839242576247

In [99]:
import ta 


bb = ta.volatility.BollingerBands(close=merged_data['close'])





hband
False    31
True     19
Name: count, dtype: int64